# **Longstaff-Schwarz Method for American Option Pricing**

The Longstaff–Schwarz method estimates the price of an American option by combining Monte Carlo simulation with backward induction.
First, a large number of asset price paths is generated under the Black–Scholes model. At maturity, the option value is equal to its payoff.
Then, moving backward in time, the algorithm compares the immediate exercise value with the continuation value, that is, the expected discounted future payoff conditional on the current asset price.

For each exercise date, the continuation value is approximated by regression on the in-the-money paths. In this implementation, a quadratic polynomial is used to approximate

$$
C_t(S_t)\approx \mathbb{E}\left[e^{-r\Delta t}V_{t+\Delta t}\mid S_t\right].
$$

The option is exercised whenever the intrinsic value is greater than the estimated continuation value. In this way, the method provides an approximation of the optimal early exercise strategy and therefore of the American option price.

In [ ]:
import numpy as np

In [ ]:
def longstaff_schwarz(S_0, K, sigma, r, T, N, n, option_type="put"):
  dt=T/n
  S=np.zeros((N, n+1))
  S[:,0]=S_0
  for i in range(1, n+1):
    Z=np.random.standard_normal(N)
    S[:,i]=S[:,i-1]*np.exp((r-0.5*sigma**2)*dt+sigma*np.sqrt(dt)*Z)

  if option_type=="call":
    payoff=np.maximum(S[:,n]-K,0)
  elif option_type=="put":
    payoff=np.maximum(K-S[:,n],0)
  else:
    raise ValueError("option_type should be 'call' or 'put' ")

  for i in range(n-1,0,-1):
    if option_type=="call":
      intrinsic_value=np.maximum(S[:,i]-K,0)
    else:
      intrinsic_value=np.maximum(K-S[:,i],0)

    payoff=payoff*np.exp(-r*dt)
    itm=intrinsic_value>0

    X=S[itm,i]
    Y=payoff[itm]
    c=np.polyfit(X,Y,2)
    continuation_value=np.polyval(c,X)

    exercise=intrinsic_value[itm]>continuation_value

    update=np.where(itm)[0]
    payoff[update[exercise]]=intrinsic_value[update[exercise]]

  return np.mean(payoff)


# Numerical example

In [ ]:
S_0=100
K=100
sigma=0.2
r=0.02
T=2
N=100000
n=500

In [ ]:
price=longstaff_schwarz(S_0, K, sigma, r, T, N, n, option_type="put")
print("American Put Price:", price)

American Put Price: 9.503975285979484
